In [1]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="w9voyLSJI3cKT65JIE2A")
project = rf.workspace("dataset-bounding-box").project("instance-segmentation-dataset-hpuah")
version = project.version(4)
dataset = version.download("coco-segmentation")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.0/250.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 69.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 102.0 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Uninstalling idna-3.11:
      Successfully uninstalled idna-3.11
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0,


Extracting Dataset Version Zip to instance-segmentation-dataset-4 in coco-segmentation:: 100%|██████████| 3123/3123 [00:00<00:00, 5313.04it/s]


In [2]:
import os
import json
import shutil
import numpy as np
import cv2

try:
    from pycocotools import mask as mask_utils
except ImportError:
    mask_utils = None
    print("pycocotools not available — RLE segmentations will be skipped.")

def coco_to_yolo_seg(coco_json_path, images_src_dir, out_root, split, ignore_name=None):
    with open(coco_json_path) as f:
        coco = json.load(f)
    images = {img['id']: img for img in coco['images']}
    filtered_categories = [c for c in coco['categories'] if c['name'] != ignore_name]
    filtered_categories = sorted(filtered_categories, key=lambda c: c['id'])
    cat_id_to_yolo = {c['id']: i for i, c in enumerate(filtered_categories)}
    class_names = [c['name'] for c in filtered_categories]

    out_img_dir = os.path.join(out_root, 'images', split)
    out_lbl_dir = os.path.join(out_root, 'labels', split)
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_lbl_dir, exist_ok=True)

    anns_by_image = {}
    for ann in coco['annotations']:
        if ann['category_id'] in cat_id_to_yolo:
            anns_by_image.setdefault(ann['image_id'], []).append(ann)

    for img_id, img_info in images.items():
        file_name = img_info['file_name']
        w, h = img_info['width'], img_info['height']
        src_img_path = os.path.join(images_src_dir, file_name)
        if not os.path.exists(src_img_path): continue
        dst_img_path = os.path.join(out_img_dir, os.path.basename(file_name))
        if not os.path.exists(dst_img_path): shutil.copy2(src_img_path, dst_img_path)

        lines = []
        for ann in anns_by_image.get(img_id, []):
            seg = ann.get('segmentation')
            if not seg: continue
            cls = cat_id_to_yolo[ann['category_id']]
            polygons = []
            if isinstance(seg, list):
                for poly in seg:
                    if len(poly) >= 6: polygons.append(poly)
            if not polygons: continue
            merged = []
            for poly in polygons: merged.extend(poly)
            norm = []
            for i in range(0, len(merged), 2):
                x = min(max(merged[i] / w, 0.0), 1.0)
                y = min(max(merged[i + 1] / h, 0.0), 1.0)
                norm.append(f"{x:.6f}")
                norm.append(f"{y:.6f}")
            lines.append(f"{cls} " + " ".join(norm))

        label_name = os.path.splitext(os.path.basename(file_name))[0] + '.txt'
        with open(os.path.join(out_lbl_dir, label_name), 'w') as f:
            f.write("\n".join(lines))
    return class_names

COCO_ROOT = '/kaggle/working/instance-segmentation-dataset-4'
OUT_ROOT  = '/kaggle/working/CaneScan_Autolabeled-3'  
IGNORE_CATEGORY_NAME = 'instance-segmentation-dataset'
splits = {'train': ('train/_annotations.coco.json', 'train'), 'valid': ('valid/_annotations.coco.json', 'valid')}

class_names = None
for split, (ann_rel, img_subdir) in splits.items():
    ann_path = os.path.join(COCO_ROOT, ann_rel)
    img_dir  = os.path.join(COCO_ROOT, img_subdir)
    if os.path.exists(ann_path):
        names = coco_to_yolo_seg(ann_path, img_dir, OUT_ROOT, split, ignore_name=IGNORE_CATEGORY_NAME)
        class_names = class_names or names

if class_names:
    import yaml as _yaml
    data_yaml = {'path': OUT_ROOT, 'train': os.path.join(OUT_ROOT, 'images', 'train'), 'val': os.path.join(OUT_ROOT, 'images', 'valid'), 'nc': len(class_names), 'names': class_names}
    with open(os.path.join(OUT_ROOT, 'data.yaml'), 'w') as f: _yaml.dump(data_yaml, f, sort_keys=False)

In [3]:
!pip install ultralytics -q
import ultralytics
ultralytics.checks()

Ultralytics 8.4.69 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Setup complete ✅ (4 CPUs, 31.3 GB RAM, 6960.6/8062.4 GB disk)


In [4]:
import glob
nn_dir = glob.glob('/usr/local/lib/python3*/dist-packages/ultralytics/nn')[0]
tasks_path = f'{nn_dir}/tasks.py'
custom_layers_path = f'{nn_dir}/custom_layers.py'

custom_layers_code = '''
import torch
import torch.nn as nn

class TriScaleEntry(nn.Module):
    def __init__(self, c1, c2, stride=1):
        super().__init__()
        color_ch   = max(c2 // 4, 8)
        remaining  = c2 - color_ch
        shape_ch   = remaining // 2
        texture_ch = remaining - shape_ch
        self.branch1 = nn.Sequential(nn.Conv2d(c1, shape_ch, kernel_size=3, stride=stride, padding=1, bias=False), nn.BatchNorm2d(shape_ch), nn.SiLU())
        self.branch2 = nn.Sequential(nn.Conv2d(c1, texture_ch, kernel_size=5, stride=stride, padding=2, bias=False), nn.BatchNorm2d(texture_ch), nn.SiLU())
        self.branch_color = nn.Sequential(nn.Conv2d(c1 + 2, color_ch, kernel_size=3, stride=stride, padding=1, bias=False), nn.BatchNorm2d(color_ch), nn.SiLU())
    def _with_color_indices(self, x):
        r, g, b = x[:, 0:1], x[:, 1:2], x[:, 2:3]
        return torch.cat([x, r - g, g - b], dim=1)
    def forward(self, x):
        return torch.cat([self.branch1(x), self.branch2(x), self.branch_color(self._with_color_indices(x))], dim=1)

class DualScaleEntry(nn.Module):
    def __init__(self, c1, c2, stride=1):  
        super().__init__()
        hidden_ch = c2 // 2
        self.branch1 = nn.Sequential(nn.Conv2d(c1, hidden_ch, kernel_size=3, stride=stride, padding=1, bias=False), nn.BatchNorm2d(hidden_ch), nn.SiLU())
        self.branch2 = nn.Sequential(nn.Conv2d(c1, hidden_ch, kernel_size=5, stride=stride, padding=2, bias=False), nn.BatchNorm2d(hidden_ch), nn.SiLU())
    def forward(self, x):
        return torch.cat([self.branch1(x), self.branch2(x)], dim=1)

class DetectionChannelAttention(nn.Module):
    def __init__(self, c1, c2=None):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(nn.Linear(c1, c1 // 16, bias=False), nn.ReLU(inplace=True), nn.Linear(c1 // 16, c1, bias=False), nn.Sigmoid())\n    def forward(self, x):
        b, c, _, _ = x.size()
        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)
        return x * y.expand_as(x)
'''.strip() + '\n'

with open(custom_layers_path, 'w') as f: f.write(custom_layers_code)
content = open(tasks_path).read()

old = '            A2C2f,\n        }\n    )\n    repeat_modules'
new = '            A2C2f,\n            DualScaleEntry,\n            TriScaleEntry,\n            DetectionChannelAttention,\n        }\n    )\n    repeat_modules'

if 'TriScaleEntry' not in content:
    content = content.replace(old, new, 1)

import_line = 'from ultralytics.nn.custom_layers import DualScaleEntry, TriScaleEntry, DetectionChannelAttention\n'
if import_line not in content:
    content = content.rstrip('\n') + '\n\n' + import_line

open(tasks_path, 'w').write(content)
print("✅ Core tasks.py file successfully re-patched for registration accuracy.")

✅ Core tasks.py file successfully re-patched for registration accuracy.


In [5]:
import os, sys, time, threading, psutil, torch
def memory_circuit_breaker(ram_threshold_pct=93.0, vram_threshold_pct=93.0, check_interval_sec=3):
    def monitor():
        while True:
            ram_pct = psutil.virtual_memory().percent
            vram_pct = 0.0
            if torch.cuda.is_available():
                for i in range(torch.cuda.device_count()):
                    vram_pct = max(vram_pct, (torch.cuda.memory_allocated(i) / torch.cuda.get_device_properties(i).total_memory) * 100)
            if ram_pct >= ram_threshold_pct or vram_pct >= vram_threshold_pct:
                os._exit(0)
            time.sleep(check_interval_sec)
    threading.Thread(target=monitor, daemon=True).start()
memory_circuit_breaker()

In [6]:
# ── Cell 7: Multi-GPU Checkpoint Resumption (With Custom Data Overrides) ──
import os
import torch

LAST_CHECKPOINT = '/kaggle/input/models/melchoestanilsmurf3/epoch415/transformers/default/1/last(1).pt'
DATA_YAML_PATH  = '/kaggle/working/CaneScan_Autolabeled-3/data.yaml'  # Points directly to your custom 6-class map

if not os.path.exists(LAST_CHECKPOINT):
    raise FileNotFoundError(
        f"❌ Checkpoint file not found at '{LAST_CHECKPOINT}'. "
        f"Please verify where your last.pt file is saved."
    )

if not os.path.exists(DATA_YAML_PATH):
    raise FileNotFoundError(
        f"❌ Remapped dataset configuration file not found at '{DATA_YAML_PATH}'. "
        f"Please ensure Cell 5 ran successfully before executing this block."
    )

num_gpus = torch.cuda.device_count()
print("=== Hardware Detection ===")
print(f"Number of available processing nodes: {num_gpus}")

if num_gpus > 1:
    print(f"\n🚀 Multi-GPU environment detected ({num_gpus} T4 GPUs).")
    devices = ",".join([str(i) for i in range(num_gpus)])
    
    # We use f-string quotes to protect the path containing parentheses
    command = f'yolo segment train model="{LAST_CHECKPOINT}" data="{DATA_YAML_PATH}" resume=True device={devices}'
    
    print(f"Executing: {command}")
    !{command}
else:
    print("\nSingle GPU or fallback CPU processing layout detected. Initializing standard tracking loops...")
    from ultralytics import YOLO
    model = YOLO(LAST_CHECKPOINT)
    model.train(data=DATA_YAML_PATH, resume=True)

=== Hardware Detection ===
Number of available processing nodes: 2

🚀 Multi-GPU environment detected (2 T4 GPUs).
Executing: yolo segment train model="/kaggle/input/models/melchoestanilsmurf3/epoch415/transformers/default/1/last(1).pt" data="/kaggle/working/CaneScan_Autolabeled-3/data.yaml" resume=True device=0,1
Ultralytics 8.4.69 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
                                                       CUDA:1 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=10.0, cache=False, cfg=None, classes=None, close_mosaic=30, cls=1.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/CaneScan_Autolabeled-3/data.yaml, degrees=15.0, deterministic=True, device=0,1, dfl=2.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, 